# Lesson 4.2 — Handoff Contracts

Two contracts on the real stack: (a) the reference → controller handoff must carry the feed-forward derivatives (full beats feedback-only); (b) the twist ↔ joint-rate relation ξ = J q̇ round-trips in a consistent frame.

In [1]:
import numpy as np
from modules.module09.integration import (GreenhouseWorld, model_perception,
    understand, to_configuration, plan_reference, execute_reference,
    velocity_layer, geometric_jacobian, fk_xy, P2, T2)
checks = []
world = GreenhouseWorld.demo_row(n=6, seed=1)
dets = model_perception(world, rng=np.random.default_rng(7))
_, tgt = understand(dets, world)
layer = plan_reference(world.q.copy(), to_configuration(tgt), rng=np.random.default_rng(0))


### Contract (a): full feed-forward tracks tighter than feedback-only

In [2]:
full = execute_reference(layer, ff='full')
fb_only = execute_reference(layer, ff='none')
print('rms  full feed-forward: %.5f' % full['rms'])
print('rms  feedback-only    : %.5f' % fb_only['rms'])
checks.append(full['rms'] < fb_only['rms'])   # the full handoff contract tracks better


rms  full feed-forward: 0.00007
rms  feedback-only    : 0.07772


### Contract (b): ξ = J q̇ round-trips (away from singularity)

In [3]:
q = np.array([0.4, 0.9])                       # a well-conditioned configuration
qd_true = np.array([0.3, -0.2])
Jv = geometric_jacobian(P2, T2, q)[:2, :]      # planar tool twist = linear part
xi_d = Jv @ qd_true                            # joint rate -> tool twist
qd_back, info = velocity_layer(P2, T2, q, xi_d)  # tool twist -> joint rate (M6)
print('q_dot true:', np.round(qd_true,3), '| recovered:', np.round(qd_back,3))
print('sigma_min: %.3f (not singular: %s)' % (info['sigma_min'], not info['singular']))
checks.append(np.allclose(qd_true, qd_back, atol=1e-6))   # consistent frame -> round-trips
checks.append(not info['singular'])
assert all(checks), f'FAILED: {checks}'
print(f'{sum(checks)}/{len(checks)} checks passed.')
print('All checks passed.')


q_dot true: [ 0.3 -0.2] | recovered: [ 0.3 -0.2]
sigma_min: 0.137 (not singular: True)
3/3 checks passed.
All checks passed.
